# Predição de óbito ou readmissão em 6 meses em pacientes com insuficiência cardíaca

Trabalho final de Inteligência Computacional Aplicada a Problemas de Engenharia (UFMG 2026/2).

**Objetivo**: a partir das informações disponíveis na internação, prever se o paciente terá **óbito ou readmissão hospitalar em até 6 meses** (classificação binária).

Etapas:
1. **Análise exploratória dos dados (EDA)** — só observa os dados; nenhuma informação é removida ou transformada aqui.
2. **Extração e seleção de atributos** — toda remoção ou transformação de informação (colunas com vazamento, faltantes, codificação, seleção) acontece aqui.
3. Treinamento de modelos lineares e não lineares
4. Avaliação e seleção de modelos

## Sobre a base de dados

**Hospitalized patients with heart failure: integrating electronic healthcare records and external outcome data**, versão 1.3 (22/05/2022), PhysioNet.

- **Acesso**: https://physionet.org/content/heart-failure-zigong/1.3/ (acesso restrito: exige credenciamento no PhysioNet e assinatura do *PhysioNet Restricted Health Data Use Agreement 1.5.0*).
- **Origem**: prontuários eletrônicos do *Zigong Fourth People's Hospital* (Sichuan, China), internações entre **dezembro/2016 e junho/2019**.
- **População**: **2008 pacientes** internados consecutivamente com insuficiência cardíaca (IC), diagnosticada pelos critérios da *European Society of Cardiology* (sintomas/sinais de IC + BNP elevado ou evidência objetiva de alteração cardíaca estrutural/funcional).
- **Estrutura**: dados agregados em **uma linha por internação** (sem séries temporais). Estudo retrospectivo aprovado pelo comitê de ética do hospital (nº 2020-010), com dispensa de consentimento.
- **Conteúdo** (`dat.csv`): dados demográficos e de admissão, sinais vitais, classificação da IC (tipo, NYHA, Killip), comorbidades (índice de Charlson), nível de consciência (Glasgow), ecocardiograma, ~100 exames laboratoriais (hemograma, função renal e hepática, coagulação, marcadores cardíacos, eletrólitos, lipídios, gasometria) e **desfechos de seguimento**.
- **Desfechos**: óbito e readmissão hospitalar aferidos em visitas obrigatórias (ou por telefone) aos **28 dias, 3 meses e 6 meses** após a admissão, além do desfecho na própria internação e do retorno ao pronto-socorro em 6 meses.
- **Medicamentos** (`dat_md.csv`): fármacos administrados durante a internação (diuréticos, inotrópicos, vasodilatadores, anticoagulantes, antiagregantes etc.), em formato longo; entradas repetidas indicam doses repetidas, sem horário de administração.

**Como citar**

- Zhang, Z., Cao, L., Zhao, Y., Xu, Z., Chen, R., Lv, L., & Xu, P. (2022). *Hospitalized patients with heart failure: integrating electronic healthcare records and external outcome data* (version 1.3). PhysioNet. https://doi.org/10.13026/5m60-vs44
- Zhang Z, Cao L, Chen R, Zhao Y, Lv L, Xu Z, Xu P. *Electronic healthcare records and external outcome data for hospitalized patients with heart failure*. Scientific Data. 2021;8(1):46. https://doi.org/10.1038/s41597-021-00835-9
- Citação padrão do PhysioNet, conforme indicado na página do dataset.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42

DATA_DIR = Path("../DataBase")
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="paper")

SERIES_COLOR = "#2a78d6"
NEUTRAL_COLOR = "#a3a29c"
NEUTRAL_LIGHT_COLOR = "#e8e7e3"
MUTED_COLOR = "#52514e"

In [ ]:
def save_fig(fig, name):
    """Save a figure as vector PDF in FIGURES_DIR."""
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")

### Carregamento dos dados

In [ ]:
patients = pd.read_csv(DATA_DIR / "dat.csv", index_col=0)
drugs = pd.read_csv(DATA_DIR / "dat_md.csv", index_col=0)
data_dictionary = pd.read_csv(DATA_DIR / "dataDictionary.csv", index_col=0)

print(f"patients: {patients.shape}")
print(f"drugs: {drugs.shape}")
print(f"data_dictionary: {data_dictionary.shape}")

## 1. Análise exploratória dos dados

### 1.1 Visão geral

In [ ]:
patients.head()

In [ ]:
patients.dtypes.value_counts()

### 1.2 Dicionário de variáveis

As 166 colunas de `dat.csv` foram agrupadas por tema, seguindo a ordem do `dataDictionary.csv`:

| Grupo | O que contém |
|---|---|
| Identificação | Número da internação |
| Demografia | Sexo, faixa etária (só por década) e ocupação |
| Admissão e histórico | Setor de admissão, se foi emergência e nº de internações anteriores |
| Sinais vitais e antropometria | Temperatura, pulso, respiração, pressão arterial, peso, altura e IMC na admissão |
| Classificação da IC | Tipo de IC (esquerda/direita/ambas), classe funcional NYHA e classe de Killip |
| Comorbidades | Doenças prévias (binárias) e o índice de comorbidade de Charlson (CCI) |
| Estado clínico na admissão | Insuficiência respiratória tipo II, consciência, componentes da escala de Glasgow, suporte ventilatório, oxigenoterapia, FiO2 e lesão renal aguda |
| Ecocardiograma | Fração de ejeção (LVEF), diâmetro do VE, velocidades das ondas mitrais E/A e refluxo tricúspide |
| Lab. — * | Exames laboratoriais, separados por painel |
| Desfechos e seguimento | Óbito/readmissão em 28 dias, 3 e 6 meses, desfecho na internação, retorno ao PS e tempos até cada evento |
| Informações da alta | Destino da alta, setor da alta e dias de internação |

Os grupos *Identificação*, *Desfechos e seguimento* e *Informações da alta* só são conhecidos após a alta (ou não são informação clínica) e serão removidos na etapa 2.

Em *valores observados*, variáveis numéricas com mais de 10 valores distintos mostram `mínimo – mediana – máximo`; as demais listam as categorias.

In [ ]:
def column_span(first, last):
    """Columns of `patients` from `first` to `last`, inclusive, in file order."""
    return patients.loc[:, first:last].columns.tolist()


VARIABLE_GROUPS = {
    "Identificação": ["inpatient.number"],
    "Demografia": ["gender", "ageCat", "occupation"],
    "Admissão e histórico": ["admission.ward", "admission.way", "visit.times"],
    "Sinais vitais e antropometria": column_span("body.temperature", "BMI"),
    "Classificação da IC": column_span("type.of.heart.failure", "Killip.grade"),
    "Comorbidades": column_span("myocardial.infarction", "CCI.score"),
    "Estado clínico na admissão": column_span("type.II.respiratory.failure", "acute.renal.failure") + ["GCS"],
    "Ecocardiograma": column_span("LVEF", "tricuspid.valve.return.pressure"),
    "Lab. — função renal": column_span("creatinine.enzymatic.method", "cystatin"),
    "Lab. — hemograma": column_span("white.blood.cell", "neutrophil.count"),
    "Lab. — coagulação": column_span("D.dimer", "fibrinogen"),
    "Lab. — marcadores cardíacos": ["high.sensitivity.troponin", "myoglobin"]
    + column_span("creatine.kinase.isoenzyme.to.creatine.kinase", "brain.natriuretic.peptide"),
    "Lab. — eletrólitos": column_span("carbon.dioxide.binding.capacity", "serum.magnesium"),
    "Lab. — bioquímica hepática e inflamação": column_span("high.sensitivity.protein", "erythrocyte.sedimentation.rate"),
    "Lab. — lipídios": column_span("cholesterol", "lipoprotein"),
    "Lab. — gasometria": column_span("pH", "total.hemoglobin"),
    "Desfechos e seguimento": column_span("outcome.during.hospitalization", "time.to.emergency.department.within.6.months"),
    "Informações da alta": ["DestinationDischarge", "discharge.department", "dischargeDay"],
}
COLUMN_GROUP = {column: group for group, columns in VARIABLE_GROUPS.items() for column in columns}
assert sorted(COLUMN_GROUP) == sorted(patients.columns), "every column must belong to exactly one group"

In [ ]:
def variable_kind(series):
    n_unique = series.nunique()
    if n_unique == 2:
        return "binária"
    if pd.api.types.is_numeric_dtype(series):
        return "numérica" if n_unique > 10 else "discreta"
    return "categórica"


def observed_values(series):
    values = series.dropna()
    if pd.api.types.is_numeric_dtype(series) and values.nunique() > 10:
        return f"{values.min():g} – {values.median():g} – {values.max():g}"
    return ", ".join(str(v) for v in sorted(values.unique(), key=str))


descriptions = data_dictionary.set_index("Variables")["Description"]

variable_overview = pd.DataFrame(
    {
        "grupo": [COLUMN_GROUP[c] for c in patients.columns],
        "variável": patients.columns,
        "descrição": descriptions.reindex(patients.columns).values,
        "tipo": [variable_kind(patients[c]) for c in patients.columns],
        "faltantes (%)": patients.isna().mean().mul(100).round(1).values,
        "valores observados": [observed_values(patients[c]) for c in patients.columns],
    }
)
variable_overview["grupo"] = pd.Categorical(variable_overview["grupo"], categories=list(VARIABLE_GROUPS), ordered=True)
variable_overview = variable_overview.sort_values("grupo", kind="stable").reset_index(drop=True)
variable_overview.to_csv(RESULTS_DIR / "variable_overview.csv", index=False)

variable_overview.groupby("grupo", observed=True).agg(
    n_variáveis=("variável", "size"),
    faltantes_médio=("faltantes (%)", "mean"),
    faltantes_máx=("faltantes (%)", "max"),
).round(1)

In [ ]:
from IPython.display import Markdown, display

with pd.option_context("display.max_colwidth", None):
    for group, table in variable_overview.groupby("grupo", observed=True):
        display(Markdown(f"#### {group}"))
        display(table.drop(columns="grupo").set_index("variável"))

### 1.3 Medicamentos (`dat_md.csv`)

Formato longo: uma linha por par (internação, fármaco). Para cada fármaco: nº de registros e nº de internações distintas em que foi usado.

Apesar de a página do PhysioNet dizer que entradas repetidas indicam doses repetidas, nesta versão `registros == internações` para todos os fármacos — ou seja, cada fármaco aparece no máximo uma vez por internação. A informação disponível é só **usou / não usou** (indicador binário), sem dose nem frequência.

In [ ]:
drug_summary = (
    drugs.groupby("Drug_name")
    .agg(registros=("inpatient.number", "size"), internações=("inpatient.number", "nunique"))
    .assign(**{"internações (%)": lambda t: t["internações"].div(len(patients)).mul(100).round(1)})
    .sort_values("internações", ascending=False)
)
print(f"Internações com ao menos um medicamento: {drugs["inpatient.number"].nunique()} de {len(patients)}")
print(f"Internações em drugs ausentes de patients: {len(set(drugs["inpatient.number"]) - set(patients["inpatient.number"]))}")
drug_summary

### 1.4 Variável alvo: óbito ou readmissão em 6 meses

**Por que este alvo**: um teste preliminar comparou vários desfechos candidatos (readmissão em 28 dias/3/6 meses, óbito, alta a pedido, tempo de internação, classe NYHA). O composto *óbito ou readmissão em 6 meses* é o endpoint mais usado em estudos de prognóstico de IC, tem as classes mais equilibradas e evita uma distorção da readmissão pura: quem morre após a alta não pode ser readmitido e seria contado como "sem evento".

**Definição**: `target = 1` se o paciente foi readmitido em qualquer janela registrada (28 dias, 3 ou 6 meses) **ou** morreu em até 6 meses; senão `target = 0`. Usar qualquer janela de readmissão (e não só a de 6 meses) cobre 3 registros inconsistentes em que há readmissão em 3 meses mas não em 6 (ver checagem abaixo).

O gráfico a seguir mostra a prevalência de cada desfecho disponível na base, para situar o alvo escolhido em relação às alternativas: desfechos de óbito são raros demais (< 3%) para modelar com segurança, enquanto readmissão e o composto têm classes suficientemente equilibradas.

In [ ]:
BINARY_OUTCOMES = {
    "death.within.28.days": "Óbito em 28 dias",
    "death.within.3.months": "Óbito em 3 meses",
    "death.within.6.months": "Óbito em 6 meses",
    "re.admission.within.28.days": "Readmissão em 28 dias",
    "re.admission.within.3.months": "Readmissão em 3 meses",
    "re.admission.within.6.months": "Readmissão em 6 meses",
    "return.to.emergency.department.within.6.months": "Retorno ao PS em 6 meses",
}

READMISSION_COLUMNS = ["re.admission.within.28.days", "re.admission.within.3.months", "re.admission.within.6.months"]
readmitted = patients[READMISSION_COLUMNS].eq(1).any(axis=1)
died = patients["death.within.6.months"].eq(1)
target = (readmitted | died).astype(int).rename("target")

outcome_summary = pd.DataFrame(
    {
        "positivos": patients[list(BINARY_OUTCOMES)].sum().astype(int),
        "prevalência (%)": patients[list(BINARY_OUTCOMES)].mean().mul(100).round(1),
    }
).rename(index=BINARY_OUTCOMES)
outcome_summary.loc["Óbito ou readmissão em 6 meses (alvo)"] = [target.sum(), round(target.mean() * 100, 1)]
outcome_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
plot_data = outcome_summary.iloc[::-1]
colors = [SERIES_COLOR if "alvo" in label else NEUTRAL_COLOR for label in plot_data.index]
ax.barh(plot_data.index, plot_data["prevalência (%)"], color=colors, height=0.6)
for label, row in plot_data.iterrows():
    ax.text(row["prevalência (%)"] + 0.8, label, f"{row['positivos']:.0f} ({row['prevalência (%)']:.1f}%)",
            va="center", fontsize=8, color=MUTED_COLOR)
ax.set_xlabel("Prevalência (%)")
ax.set_xlim(0, plot_data["prevalência (%)"].max() * 1.25)
ax.grid(axis="y", visible=False)
save_fig(fig, "outcome_prevalence")
plt.show()

Composição do alvo e checagens de consistência. Os desfechos são cumulativos no tempo (quem teve o evento em 28 dias deve tê-lo também em 3 e 6 meses); óbito e readmissão em 6 meses não se sobrepõem nesta base. Os óbitos durante a própria internação também entram no alvo, pois estão contidos em `death.within.6.months`.

In [ ]:
for event in ("death", "re.admission"):
    d28, m3, m6 = (patients[f"{event}.within.{window}"] for window in ("28.days", "3.months", "6.months"))
    print(f"{event}: 28d sem 3m = {(d28 > m3).sum()} | 3m sem 6m = {(m3 > m6).sum()}")

print(f"\nSó readmissão: {(readmitted & ~died).sum()}")
print(f"Só óbito: {(died & ~readmitted).sum()}")
print(f"Readmissão e óbito: {(readmitted & died).sum()}")
print(f"Óbitos durante a internação: {patients['outcome.during.hospitalization'].eq('Dead').sum()}")
print(f"\nTotal no alvo: {target.sum()} de {len(target)} ({target.mean():.1%})")

### 1.5 Valores faltantes

O objetivo desta seção é entender **quanto** falta e **como** falta, porque isso define as decisões da etapa 2 (quais colunas descartar, como imputar).

Atenção: nem todo `NA` é dado perdido. Em `respiratory.support.`, por exemplo, só existem as categorias `IMV`/`NIMV`, então o `NA` provavelmente significa *sem suporte ventilatório*. Da mesma forma, os tempos até evento são `NA` quando o evento não ocorreu.

In [ ]:
missing_rate = patients.isna().mean().mul(100)

missing_bins = pd.cut(missing_rate, bins=[-0.1, 0, 20, 50, 100], labels=["0%", "(0, 20]%", "(20, 50]%", "(50, 100]%"])
print("Nº de variáveis por faixa de faltantes:")
print(missing_bins.value_counts().sort_index().to_string())
print(f"\nrespiratory.support. — faltantes: {missing_rate['respiratory.support.']:.1f}%")
print(f"Internações sem nenhum faltante: {patients.notna().all(axis=1).sum()}")

**Quanto falta em cada variável.** Cada barra é uma variável com algum faltante. As linhas tracejadas marcam 20% e 50%, limiares usuais na literatura para descartar uma variável em vez de imputá-la. Serve para escolher esse limiar na etapa 2: há um grupo grande acima de 50% (gasometria, lipídios, ecocardiograma) e quase todo o resto abaixo de 20%.

In [ ]:
missing_only = missing_rate[missing_rate > 0].sort_values()

fig, ax = plt.subplots(figsize=(7, 0.16 * len(missing_only) + 1))
ax.barh(missing_only.index, missing_only.values, color=SERIES_COLOR, height=0.7)
for threshold in (20, 50):
    ax.axvline(threshold, color=MUTED_COLOR, linestyle="--", linewidth=1)
ax.set_xlabel("Valores faltantes (%)")
ax.set_xlim(0, 100)
ax.tick_params(axis="y", labelsize=6)
ax.margins(y=0.005)
ax.grid(axis="y", visible=False)
save_fig(fig, "missing_rate_by_variable")
plt.show()

**Como falta: os faltantes aparecem juntos?** Cada linha da imagem é uma internação e cada coluna uma variável (só as com algum faltante), agrupadas por tema; azul = faltante. As internações estão ordenadas da mais completa (topo) para a menos completa.

Como ler: faixas azuis que atravessam um grupo inteiro na mesma linha indicam que **o painel de exames inteiro não foi pedido** para aquele paciente (ex.: gasometria), e não valores perdidos ao acaso. Isso importa para a etapa 2: faltante "por não ter sido pedido" pode carregar informação clínica (o médico não achou necessário), o que justifica considerar indicadores de faltante como atributo, em vez de só imputar.

In [ ]:
from matplotlib.colors import ListedColormap

missing_columns = [c for c in patients.columns if missing_rate[c] > 0]
missing_columns = sorted(missing_columns, key=lambda c: list(VARIABLE_GROUPS).index(COLUMN_GROUP[c]))
missing_matrix = patients[missing_columns].isna()
missing_matrix = missing_matrix.loc[missing_matrix.sum(axis=1).sort_values(kind="stable").index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(missing_matrix.values, aspect="auto", interpolation="nearest",
          cmap=ListedColormap([NEUTRAL_LIGHT_COLOR, SERIES_COLOR]))

group_of_column = pd.Series([COLUMN_GROUP[c] for c in missing_columns])
boundaries = np.flatnonzero(group_of_column.ne(group_of_column.shift())).tolist() + [len(missing_columns)]
for start, end in zip(boundaries[:-1], boundaries[1:]):
    if start > 0:
        ax.axvline(start - 0.5, color="white", linewidth=1.5)
ax.set_xticks([(start + end - 1) / 2 for start, end in zip(boundaries[:-1], boundaries[1:])])
ax.set_xticklabels(group_of_column.iloc[boundaries[:-1]], rotation=90, fontsize=7)
ax.set_yticks([])
ax.set_ylabel("Internações (mais completa no topo)")
ax.grid(False)
save_fig(fig, "missing_matrix")
plt.show()

## 2. Extração e seleção de atributos

### 2.1 Remoção de colunas com vazamento de informação

Saem as colunas que não estariam disponíveis no momento da predição (durante a internação) ou que não são informação clínica:
- **Desfechos e seguimento**: o próprio alvo e os demais desfechos, além dos tempos até cada evento;
- **Informações da alta**: destino da alta, setor da alta e dias de internação, todos registrados só na alta;
- **Identificação**: número da internação.

In [ ]:
LEAKAGE_COLUMNS = (
    VARIABLE_GROUPS["Desfechos e seguimento"]
    + VARIABLE_GROUPS["Informações da alta"]
    + VARIABLE_GROUPS["Identificação"]
)
features = patients.drop(columns=LEAKAGE_COLUMNS)
print(f"Removidas: {len(LEAKAGE_COLUMNS)} colunas | Restantes: {features.shape[1]} atributos candidatos")

## 3. Treinamento de modelos

## 4. Avaliação e seleção de modelos